Homework 10   
Author: Huiping Zhou   
Date: 4/19/2026

# Part 1 - Creating Streaming Data Using rate

First, I will start a new Spark session and set up a streaming DataFrame using the rate format. This provides simple, generated data that can be used to test functionality and explore streaming operations.

In [2]:
#create a spark section
from pyspark.sql import SparkSession         
spark = SparkSession.builder.getOrCreate()

# Sets up a streaming DataFrame using the rate source
rateDF = spark.readStream.format("rate").load()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 13:11:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/19 13:11:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
rateDF

DataFrame[timestamp: timestamp, value: bigint]

We can see the rateDF DataFrame contains two columns: timestamp and value, where value is generated sequentially over time.

Before starting the streaming query, we apply a sequence of transformations using functions from `pyspark.sql.functions` to the rate data. Specifically, these transformations 
- calculate the square root of the `value`    
- find mod 4 of the rate `value`  

These transformations are added to the DataFrame using withColumn, resulting in a new streaming DataFrame that includes both the original columns and the newly computed values.

Before starting the streaming query, we apply a sequence of transformations using functions from `pyspark.sql.functions` to the rate data. Specifically, these transformations 
- calculate the square root of the `value`    
- find mod 4 of the rate `value`  

These transformations are added to the DataFrame using withColumn, resulting in a new streaming DataFrame that includes both the original columns and the newly computed values.

In [4]:
from pyspark.sql.functions import col, sqrt, pmod
manipDF = rateDF.withColumn("squaredroot_value", sqrt(col("value"))) \
                            .withColumn("mod4_value", pmod(col("value"), 4))

Next, the transformed streaming DataFrame is written to an output sink. Initially, the stream is written to the `console` in `append` mode to verify that the transformations are being applied correctly. The streaming query is started and then stopped after execution.

In [5]:
writeDF = manipDF.writeStream \
                 .outputMode("append") \
                 .format("console") \
                 .start()

26/04/19 14:06:31 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-4addb5ec-a9ac-45a1-aa89-514735f30c73. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/19 14:06:31 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-----+-----------------+----------+
|timestamp|value|squaredroot_value|mod4_value|
+---------+-----+-----------------+----------+
+---------+-----+-----------------+----------+

-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+-----+-----------------+----------+
|           timestamp|value|squaredroot_value|mod4_value|
+--------------------+-----+-----------------+----------+
|2026-04-19 14:06:...|    0|              0.0|         0|
+--------------------+-----+-----------------+----------+

-------------------------------------------
Batch: 2
-------------------------------------------
+--------------------+-----+-----------------+----------+
|           timestamp|value|squaredroot_value|mod4_value|
+--------------------+-----+-----------------+----------+
|2026-04-19 14:06:...|    1|              1.0|         

In [6]:
writeDF.stop()

26/04/19 14:06:40 WARN DAGScheduler: Failed to cancel job group a0b0331b-3032-4e2d-ada6-78bb1924b4a8. Cannot find active jobs for it.
26/04/19 14:06:40 WARN DAGScheduler: Failed to cancel job group a0b0331b-3032-4e2d-ada6-78bb1924b4a8. Cannot find active jobs for it.


Finally, the transformed streaming data is written to an in-memory table by configuring the stream with the `memory` format and assigning a query name `my_table`. This allows the streaming results to be queried using Spark SQL after the stream has run. 

In [7]:
writeTable = manipDF.writeStream \
                    .format("memory") \
                    .queryName("my_table") \
                    .start()

26/04/19 14:06:45 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-36cb5474-ecc8-4995-bc57-6bb91194cf01. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/19 14:06:45 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Let it run for about 30 seconds and then stop the query. 

In [8]:
writeTable.stop()

26/04/19 14:07:52 WARN DAGScheduler: Failed to cancel job group c2cbf1af-09b7-4a85-afff-a6339fb6ece2. Cannot find active jobs for it.
26/04/19 14:07:52 WARN DAGScheduler: Failed to cancel job group c2cbf1af-09b7-4a85-afff-a6339fb6ece2. Cannot find active jobs for it.


Now there is a table that exists for us called my_table. We can use spark.sql() query to access it and display the full contents of the in-memory table .

In [9]:
spark.sql("select * from my_table").show()

+--------------------+-----+------------------+----------+
|           timestamp|value| squaredroot_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-19 14:06:...|    0|               0.0|         0|
|2026-04-19 14:06:...|    1|               1.0|         1|
|2026-04-19 14:06:...|    2|1.4142135623730951|         2|
|2026-04-19 14:06:...|    3|1.7320508075688772|         3|
|2026-04-19 14:06:...|    4|               2.0|         0|
|2026-04-19 14:06:...|    5|  2.23606797749979|         1|
|2026-04-19 14:06:...|    6| 2.449489742783178|         2|
|2026-04-19 14:06:...|    7|2.6457513110645907|         3|
|2026-04-19 14:06:...|    8|2.8284271247461903|         0|
|2026-04-19 14:06:...|    9|               3.0|         1|
|2026-04-19 14:06:...|   10|3.1622776601683795|         2|
|2026-04-19 14:06:...|   11|   3.3166247903554|         3|
|2026-04-19 14:06:...|   12|3.4641016151377544|         0|
|2026-04-19 14:06:...|   13| 3.605551275463989|         